<a href="https://colab.research.google.com/github/luciacardozo472/TRABAJOP_IA/blob/main/sistema_multiagentes__.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sistema Multi-Agente: 3 Agentes ML

**Arquitectura:**
- **Agente 1 - Normalizador:** limpia, imputa, escala, codifica
- **Agente 2 - Entrenador:** valida, entrena, selecciona modelo con embeddings/transformers
- **Agente 3 - Comunicador:** genera reporte en lenguaje natural

**Specs:** Transformers + Embeddings | Sin RAG | Dataset CSV 500 filas

## Instalacion de dependencias

In [ ]:
!pip install -q transformers sentence-transformers scikit-learn pandas numpy torch

## Dataset CSV - Empleados (500 filas, sin normalizar)

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

departamentos = ['IT', 'Ventas', 'RRHH', 'Gerencia', 'Marketing', 'Finanzas']
satisfacciones = ['alta', 'media', 'baja']

edades = np.random.randint(22, 60, n)
experiencias = np.random.randint(0, 35, n)
departamento = np.random.choice(departamentos, n)

# Salario correlacionado con experiencia y departamento
salario_base = {'IT': 55000, 'Ventas': 38000, 'RRHH': 42000,
                'Gerencia': 90000, 'Marketing': 45000, 'Finanzas': 60000}
salarios = np.array([salario_base[d] + experiencias[i] * 1500 + np.random.randint(-5000, 5000)
                     for i, d in enumerate(departamento)])

# Satisfaccion correlacionada con salario
satisfaccion = []
for s in salarios:
    if s > 75000:
        satisfaccion.append(np.random.choice(['alta', 'media'], p=[0.75, 0.25]))
    elif s > 50000:
        satisfaccion.append(np.random.choice(['alta', 'media', 'baja'], p=[0.4, 0.4, 0.2]))
    else:
        satisfaccion.append(np.random.choice(['media', 'baja'], p=[0.35, 0.65]))

df_raw = pd.DataFrame({
    'edad': edades,
    'salario': salarios,
    'departamento': departamento,
    'experiencia': experiencias,
    'satisfaccion': satisfaccion
})

# Introducir valores nulos (~5%)
for col in ['salario', 'experiencia', 'satisfaccion']:
    idx = np.random.choice(df_raw.index, size=int(n * 0.05), replace=False)
    df_raw.loc[idx, col] = np.nan

print(f'Dataset generado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas')
print(f'Valores nulos: {df_raw.isnull().sum().sum()}')
print(f'Distribucion satisfaccion:')
print(df_raw['satisfaccion'].value_counts())
df_raw.head(10)

---
## AGENTE 1 - Normalizador

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

class AgenteNormalizador:
    def __init__(self):
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.log = []

    def limpiar(self, df):
        nulos = df.isnull().sum().sum()
        df = df.drop_duplicates()
        self.log.append(f'[Limpieza] Nulos encontrados: {nulos}')
        return df

    def imputar(self, df):
        for col in df.columns:
            if df[col].isnull().any():
                if df[col].dtype in ['float64', 'int64']:
                    val = df[col].median()
                    df[col] = df[col].fillna(val)
                    self.log.append(f'[Imputacion] {col} -> mediana ({val:.1f})')
                else:
                    val = df[col].mode()[0]
                    df[col] = df[col].fillna(val)
                    self.log.append(f'[Imputacion] {col} -> moda ({val})')
        return df

    def codificar(self, df, cols):
        for col in cols:
            le = LabelEncoder()
            df[col + '_enc'] = le.fit_transform(df[col].astype(str))
            self.label_encoders[col] = le
            self.log.append(f'[Codificacion] {col} -> clases: {list(le.classes_)}')
        return df

    def escalar(self, df, cols):
        df[cols] = self.scaler.fit_transform(df[cols])
        self.log.append(f'[Escalado] StandardScaler en: {cols}')
        return df

    def procesar(self, df):
        print('=' * 55)
        print('AGENTE 1 - Normalizador')
        print('=' * 55)
        df = self.limpiar(df.copy())
        df = self.imputar(df)
        df = self.codificar(df, ['departamento', 'satisfaccion'])
        df = self.escalar(df, ['edad', 'salario', 'experiencia'])
        for e in self.log:
            print(' ', e)
        print(f'\nDataset limpio: {df.shape}')
        return df

agente1 = AgenteNormalizador()
df_limpio = agente1.procesar(df_raw)
df_limpio[['edad', 'salario', 'experiencia', 'departamento_enc', 'satisfaccion_enc']].head()

---
## AGENTE 2 - Entrenador

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

class AgenteEntrenador:
    def __init__(self, modelo_emb='all-MiniLM-L6-v2'):
        print('=' * 55)
        print('AGENTE 2 - Entrenador')
        print('=' * 55)
        print(f'  Cargando embedder: {modelo_emb}')
        self.embedder = SentenceTransformer(modelo_emb)
        self.mejor_modelo = None
        self.mejor_score = 0
        self.metricas = {}

    def generar_embeddings(self, df_orig):
        print('\n  [Embeddings] Generando representaciones semanticas...')
        textos = df_orig.apply(
            lambda r: f"empleado de {r['departamento']}, satisfaccion {r['satisfaccion']}, experiencia aproximada",
            axis=1
        ).tolist()
        emb = self.embedder.encode(textos, show_progress_bar=True)
        print(f'  [Embeddings] Shape: {emb.shape}')
        return emb

    def construir_features(self, df, emb):
        cols = ['edad', 'salario', 'experiencia', 'departamento_enc']
        X = np.hstack([df[cols].values, emb])
        print(f'  [Features] {len(cols)} numericas + {emb.shape[1]} embedding = {X.shape[1]} total')
        return X

    def entrenar(self, X, y):
        print('\n  [Entrenamiento] Validacion cruzada cv=5:')
        candidatos = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=500, random_state=42)
        }
        for nombre, modelo in candidatos.items():
            scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
            m, s = scores.mean(), scores.std()
            self.metricas[nombre] = {'accuracy_media': round(m, 4), 'std': round(s, 4)}
            print(f'    {nombre}: {m:.4f} +/- {s:.4f}')
            if m > self.mejor_score:
                self.mejor_score = m
                self.mejor_nombre = nombre
                self.mejor_modelo = modelo
        self.mejor_modelo.fit(X, y)
        print(f'\n  Mejor modelo: {self.mejor_nombre} ({self.mejor_score:.4f})')

    def procesar(self, df, df_orig):
        emb = self.generar_embeddings(df_orig)
        X = self.construir_features(df, emb)
        y = df['satisfaccion_enc'].values
        self.entrenar(X, y)
        return {
            'modelo': self.mejor_modelo,
            'nombre_modelo': self.mejor_nombre,
            'metricas': self.metricas,
            'mejor_accuracy': self.mejor_score,
            'X': X, 'y': y
        }

agente2 = AgenteEntrenador()
resultado = agente2.procesar(df_limpio, df_raw)